# Dynamic Line Ratings (DLR)

> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerSimulations.jl/PowerSimulations.jl/docs/Project.toml`
 >   [336ed68f] CSV v0.10.16
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.6
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [35a29f4d] DocumenterTools v0.1.21
 >   [87dc4568] HiGHS v1.24.1
 >   [fc1677e0] HydroPowerSimulations v0.19.0
 >   [2cd47ed4] InfrastructureSystems v3.6.3
 >   [4076af6c] JuMP v1.31.1
 >   [23fbe1c1] Latexify v0.16.11
 >   [98b081ad] Literate v2.21.0
 >   [94fada2c] PowerFlows v0.25.0
 >   [bed98974] PowerNetworkMatrices v0.24.3
 >   [e690365d] PowerSimulations v0.38.2 `~/work/PowerSimulations.jl/PowerSimulations.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.6.0
 >   [bcd98974] PowerSystems v5.12.1
 >   [08abe8d2] PrettyTables v3.4.5
 > ⌅ [9e3dc215] TimeSeries v0.25.2
 >   [f43a241f] Downloads v1.7.0
 > Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status --outdated`
 > 
 > ```


## Introduction

Static branch ratings use a fixed thermal limit $R^\text{max}$ for each transmission
line. **Dynamic Line Ratings (DLR)** replace this fixed limit with a time-varying
parameter $R^\text{max}_t$, allowing the optimizer to exploit periods when ambient
conditions (wind, temperature) permit higher line flows. This reduces curtailment and
can lower total generation cost compared to conservative static limits.

This tutorial demonstrates how to:

1. Attach a DLR time series to transmission branches in a `PowerSystems.System`.
2. Build a `PTDFPowerModel` template that activates the DLR constraints.
3. Run a multi-step simulation and read the resulting line flows and DLR parameters.

> *Note*
>
>
> Dynamic Line Ratings are supported for the `StaticBranch` (or
> `SecurityConstrainedStaticBranch`) formulation combined with a
> `PTDFPowerModel` (or any `AbstractPTDFModel`), a DC power flow
> (`DCPPowerModel` / any `PM.AbstractActivePowerModel`), or full AC
> (`ACPPowerModel` / any `PM.AbstractPowerModel`) network model.
> With `StaticBranchUnbounded` the formulation does not enforce flow
> limits, so a time-varying rating would have no effect: template
> validation emits a warning and the branch rating time series is ignored
> (the model still builds).

### Load packages

In [ ]:
using PowerSystems
using PowerSimulations
using HydroPowerSimulations
using PowerNetworkMatrices
using PowerSystemCaseBuilder
using HiGHS
using Dates
using TimeSeries

### Optimizer

In [ ]:
solver = optimizer_with_attributes(HiGHS.Optimizer, "mip_rel_gap" => 0.01)

## Data

> *Note*
>
>
> [PowerSystemCaseBuilder.jl](https://github.com/Sienna-Platform/PowerSystemCaseBuilder.jl)
> is a helper library that makes it easier to reproduce examples in the documentation
> and tutorials. Normally you would pass your local files to create the system data
> instead of calling `build_system`. For more details visit
> [PowerSystemCaseBuilder Documentation](https://sienna-platform.github.io/PowerSystems.jl/stable/how_to/powersystembuilder/)

In [ ]:
sys = build_system(PSISystems, "modified_RTS_GMLC_DA_sys")

## Preparing DLR Time Series

DLR is represented in `PowerSystems.jl` as a `SingleTimeSeries` (or `Deterministic`)
attached directly to each branch component. The time series values are scaling factors
applied to the branch's static `get_rating`. A value of `1.15` means the line can carry
15% more than its rated static capacity during that hour; a value of `0.95` represents a
de-rating.

The helper function below iterates over a list of branch names, constructs a
`SingleTimeSeries` from a vector of hourly scaling factors, and attaches it to each
branch with `scaling_factor_multiplier = get_rating` so PowerSimulations knows how to
convert the factor to a per-unit limit.

In [ ]:
function add_dlr_to_system_branches!(
    sys::System,
    branches_dlr::Vector{String},
    n_steps::Int,
    dlr_factors::Vector{Float64};
    initial_date::String = "2020-01-01",
)
    for branch_name in branches_dlr
        branch = get_component(ACTransmission, sys, branch_name)

        data_ts = collect(
            DateTime("$initial_date 0:00:00", "y-m-d H:M:S"):Hour(1):(
                DateTime("$initial_date 23:00:00", "y-m-d H:M:S") + Day(n_steps - 1)
            ),
        )

        dlr_data = TimeArray(data_ts, dlr_factors)

        PowerSystems.add_time_series!(
            sys,
            branch,
            PowerSystems.SingleTimeSeries(
                "dynamic_line_ratings",
                dlr_data;
                scaling_factor_multiplier = get_rating,
            ),
        )
    end
end

Define DLR scaling factors. Here we use a daily cycle of four blocks repeated across the
simulation horizon: the early morning hours are de-rated (0.95), mid-day has higher
capacity (1.15 and 1.05), and evening hours have intermediate capacity (0.95).

In [ ]:
n_steps = 2       # simulation length in days
initial_date = "2020-01-01"
data_days = 366   # length of DLR time series in days; must span the system's TS window

dlr_factors_daily = vcat([fill(x, 6) for x in [1.15, 1.05, 0.95, 0.95]]...)  # 24 values
dlr_factor_ts = repeat(dlr_factors_daily, data_days)

Select the branch names that will receive DLR time series. These names must match
branches present in the system.

In [ ]:
branches_dlr = [
    "A2", "AB1", "A24", "B10", "B18", "CA-1", "C22", "C34",
    "A7", "A17", "B14", "B15", "C7", "C17",
]

add_dlr_to_system_branches!(sys, branches_dlr, data_days, dlr_factor_ts; initial_date)

Because the simulation uses a rolling horizon of 48 hours (2 days), we transform the
`SingleTimeSeries` into `Deterministic` forecasts with a 48-hour horizon and a 24-hour
interval between forecast windows.

In [ ]:
transform_single_time_series!(sys, Hour(48), Day(1))

## Define the Problem Template

The template must use `PTDFPowerModel` to enable DLR constraints. The key step is
constructing `DeviceModel` with `time_series_names` that maps
`BranchRatingTimeSeriesParameter` to the time series name `"dynamic_line_ratings"`
attached to the branches above.

> *Tip*
>
>
> Any branch type that has the `"dynamic_line_ratings"` time series attached and is
> configured with `BranchRatingTimeSeriesParameter` in `time_series_names` will
> have time-varying flow limits. Branches without the time series attached will fall
> back to static limits automatically.

In [ ]:
template_uc = ProblemTemplate(
    NetworkModel(
        PTDFPowerModel;
        reduce_radial_branches = false,
        use_slacks = false,
        PTDF_matrix = PTDF(sys),
    ),
)

### Branch models with DLR enabled

In [ ]:
line_device_model = DeviceModel(
    Line,
    StaticBranch;
    time_series_names = Dict(
        BranchRatingTimeSeriesParameter => "dynamic_line_ratings",
    ),
)

tap_transformer_device_model = DeviceModel(
    TapTransformer,
    StaticBranch;
    time_series_names = Dict(
        BranchRatingTimeSeriesParameter => "dynamic_line_ratings",
    ),
)

set_device_model!(template_uc, line_device_model)
set_device_model!(template_uc, tap_transformer_device_model)

### Injection device models

In [ ]:
set_device_model!(template_uc, ThermalStandard, ThermalStandardUnitCommitment)
set_device_model!(template_uc, RenewableDispatch, RenewableFullDispatch)
set_device_model!(template_uc, RenewableNonDispatch, FixedOutput)
set_device_model!(template_uc, PowerLoad, StaticPowerLoad)
set_device_model!(template_uc, HydroDispatch, HydroDispatchRunOfRiver)
set_device_model!(
    template_uc,
    DeviceModel(TwoTerminalGenericHVDCLine, HVDCTwoTerminalLossless),
)

### Reserve models

In [ ]:
set_service_model!(template_uc, ServiceModel(VariableReserve{ReserveUp}, RangeReserve))
set_service_model!(template_uc, ServiceModel(VariableReserve{ReserveDown}, RangeReserve))

## Build and Run a Simulation

We wrap the `DecisionModel` in a `Simulation` to run multiple steps. Each step solves a
48-hour unit commitment problem and advances the clock by 24 hours.

In [ ]:
model = DecisionModel(
    template_uc,
    sys;
    name = "UC",
    optimizer = solver,
    initialize_model = true,
    store_variable_names = true,
)

models = SimulationModels(; decision_models = [model])

sequence = SimulationSequence(;
    models = models,
    ini_cond_chronology = InterProblemChronology(),
)

sim = Simulation(;
    name = "DLR_example",
    steps = n_steps,
    models = models,
    initial_time = DateTime(initial_date * "T00:00:00"),
    sequence = sequence,
    simulation_folder = mktempdir(; cleanup = true),
)

build!(sim)

execute!(sim)

## Inspecting Results

### Line flows

Retrieve the realized active power flows for `Line` and `TapTransformer` branches. Each
column corresponds to one branch; each row to one time step.

In [ ]:
results = SimulationResults(sim)
uc_results = get_decision_problem_results(results, "UC")

line_flows = read_realized_expression(
    uc_results,
    "PTDFBranchFlow__Line";
    table_format = TableFormat.WIDE,
)

transformer_flows = read_realized_expression(
    uc_results,
    "PTDFBranchFlow__TapTransformer";
    table_format = TableFormat.WIDE,
)

### DLR parameter values

The DLR parameters that were applied at each time step can be read back from the results.
The values are in per-unit (MW if multiplied by base power) and already account for the
`scaling_factor_multiplier = get_rating` applied when the time series was attached.

In [ ]:
dlr_params = read_parameter(
    uc_results,
    "BranchRatingTimeSeriesParameter__Line";
    table_format = TableFormat.WIDE,
)
first(keys(dlr_params))

> *Tip*
>
>
> To verify that DLR constraints are binding, compare the line flows in `line_flows`
> against the corresponding DLR parameter values in `dlr_params`. When demand is high
> and the DLR limit is tight, the flow should be at or near the DLR limit rather than
> the static rating.